In [1]:
# read data yearky in pickle file
import pickle
import sys
import os
import numpy as np
maindir = os.getcwd()
sys.path.append(maindir+"/src")

with open('data_yearly.pickle', 'rb') as f:
    data = pickle.load(f)


data_anomaly_yearly = data['dic_data_yearly']
data_forced_response_yearly = data['dic_forced_response_yearly']

In [2]:
import torch

def forward_diffusion_process(x, betas, steps):
    """
    Forward process that adds noise to data over specified steps.
    
    Args:
        x (torch.Tensor): Input tensor (e.g., images).
        betas (torch.Tensor): Noise variance schedule.
        steps (int): Number of diffusion steps.
        
    Returns:
        torch.Tensor: Noisy tensor after `steps` iterations.
    """
    noise = torch.randn_like(x)
    for t in range(steps):
        beta_t = betas[t]
        x = torch.sqrt(1 - beta_t) * x + torch.sqrt(beta_t) * noise
    return x

# Usage example:
betas = torch.linspace(0.0001, 0.02, steps=1000)  # Defining a linear noise schedule
x = torch.rand((1, 3, 64, 64))  # Random sample, e.g., a 64x64 image
noisy_x = forward_diffusion_process(x, betas, steps=1000)

In [4]:
def reverse_diffusion_step(x, predicted_noise, beta_t):
    """
    Single reverse diffusion step to denoise data.
    
    Args:
        x (torch.Tensor): Noisy tensor at step `t`.
        predicted_noise (torch.Tensor): Noise predicted by the network.
        beta_t (float): Noise variance at step `t`.
        
    Returns:
        torch.Tensor: Denoised tensor at step `t-1`.
    """
    return (x - torch.sqrt(beta_t) * predicted_noise) / torch.sqrt(1 - beta_t)

# Example usage:
beta_t = torch.tensor(0.01)  # Example beta value
predicted_noise = torch.randn_like(noisy_x)  # Simulated prediction from a neural network
denoised_x = reverse_diffusion_step(noisy_x, predicted_noise, beta_t)

In [5]:
def check_cuda():
    """
    Checks if CUDA is available and returns device type.
    """
    if torch.cuda.is_available():
        print("CUDA is available. Device:", torch.cuda.get_device_name(0))
        return torch.device("cuda")
    else:
        print("CUDA is not available. Running on CPU.")
        return torch.device("cpu")

device = check_cuda()

CUDA is not available. Running on CPU.


In [6]:
def forward_diffusion(x, betas, num_steps):
    """
    Adds noise to input data in a stepwise manner, based on a predefined schedule.

    Args:
        x (torch.Tensor): The original input data (e.g., image tensor).
        betas (torch.Tensor): The schedule of noise values (betas).
        num_steps (int): The number of diffusion steps.
    
    Returns:
        torch.Tensor: Noisy version of x at the end of the diffusion process.
    """
    for t in range(num_steps):
        beta_t = betas[t]  # Noise level at step t
        noise = torch.randn_like(x)  # Generate Gaussian noise
        x = torch.sqrt(1 - beta_t) * x + torch.sqrt(beta_t) * noise  # Apply noise
    return x

In [7]:
def reverse_diffusion_step(x, predicted_noise, beta_t):
    """
    Performs a single reverse diffusion step to denoise the input.

    Args:
        x (torch.Tensor): Noisy tensor at step `t`.
        predicted_noise (torch.Tensor): Noise predicted by the neural network.
        beta_t (float): Noise variance at step `t`.
    
    Returns:
        torch.Tensor: Denoised tensor for step `t-1`.
    """
    return (x - torch.sqrt(beta_t) * predicted_noise) / torch.sqrt(1 - beta_t)

In [24]:
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, input_channels, num_steps):
        super(UNet, self).__init__()
        
        # Define the encoder layers
        self.encoder1 = self.conv_block(input_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        
        # Define the decoder layers
        self.decoder1 = self.conv_block(256, 128)
        self.decoder2 = self.conv_block(128, 64)
        self.decoder3 = self.conv_block(64, input_channels)
        
        # Timestep embedding layer
        self.time_embed = nn.Linear(num_steps, 256)

    def conv_block(self, in_channels, out_channels):
        # Convolution block with ReLU and batch normalization
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x, t):
        """
        Forward pass of U-Net with timestep embedding.
        
        Args:
            x (torch.Tensor): Noisy input data.
            t (torch.Tensor): Timestep encoding.
        
        Returns:
            torch.Tensor: Predicted noise at this timestep.
        """
        # Apply timestep embedding
        t_embed = self.time_embed(t.to(torch.float32)).unsqueeze(-1).unsqueeze(-1)  # Reshape for broadcasting
        
        # Encoder pass
        e1 = self.encoder1(x + t_embed)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        
        # Decoder pass with skip connections
        d1 = self.decoder1(e3 + e2)
        d2 = self.decoder2(d1 + e1)
        d3 = self.decoder3(d2)
        
        return d3

In [25]:
import torch.optim as optim

def diffusion_loss(predicted_noise, true_noise):
    """
    Calculates the MSE loss between the predicted and true noise.
    
    Args:
        predicted_noise (torch.Tensor): Noise predicted by the neural network.
        true_noise (torch.Tensor): Actual noise added to the input.
    
    Returns:
        torch.Tensor: Mean squared error loss.
    """
    return torch.nn.functional.mse_loss(predicted_noise, true_noise)

In [34]:
def train_diffusion_model(model, dataloader, betas, num_steps, optimizer, device):
    """
    Training loop for the diffusion model.
    
    Args:
        model (torch.nn.Module): The neural network (e.g., U-Net) predicting noise.
        dataloader (torch.utils.data.DataLoader): Dataloader for training data.
        betas (torch.Tensor): Noise variance schedule.
        num_steps (int): Total number of diffusion steps.
        optimizer (torch.optim.Optimizer): Optimizer.
        device (torch.device): Device to train on.
    """
    model.train()  # Set the model to training mode
    for epoch in range(num_epochs):
        for batch in dataloader:
            x = batch[0].to(device)  # Move data to the appropriate device
            t = torch.randint(0, num_steps, (x.size(0),), device=device)  # Random timestep for each sample
            beta_t = betas[t].view(-1, 1, 1, 1)  # Extract corresponding beta for each sample
            
            # Generate noisy data
            noise = torch.randn_like(x)  # Generate Gaussian noise
            x_noisy = torch.sqrt(1 - beta_t) * x + torch.sqrt(beta_t) * noise  # Apply forward diffusion
            
            # Predict noise at this timestep
            predicted_noise = model(x_noisy, t)
            
            # Compute loss
            loss = diffusion_loss(predicted_noise, noise)
            
            # Backpropagation and optimization
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping to stabilize training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
        
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

In [35]:
def sample_diffusion(model, initial_noise, betas, num_steps, device):
    """
    Generates samples by reversing the diffusion process.
    
    Args:
        model (torch.nn.Module): Trained neural network for noise prediction.
        initial_noise (torch.Tensor): Starting point (e.g., Gaussian noise).
        betas (torch.Tensor): Noise variance schedule.
        num_steps (int): Number of diffusion steps.
        device (torch.device): Device for computation.
        
    Returns:
        torch.Tensor: Generated sample after denoising.
    """
    x = initial_noise.to(device)
    for t in reversed(range(num_steps)):
        beta_t = betas[t]
        predicted_noise = model(x, torch.tensor([t], device=device))  # Predict noise at step t
        x = (x - torch.sqrt(beta_t) * predicted_noise) / torch.sqrt(1 - beta_t)  # Reverse diffusion step
        
        # Optional: Clamp values to prevent overflow, ensuring pixel values remain within a realistic range
        x = torch.clamp(x, -1.0, 1.0)
    
    return x

In [36]:
from torch.utils.data import DataLoader, TensorDataset

# Prepare training data (assuming shape: [n_samples, years, lat, lon])
x_train = data_anomaly_yearly['CESM2']  # example: use CESM2 model
# x_train = x_train.reshape(- x_train.shape[2], x_train.shape[3])  # shape: [samples, 1, lat, lon]

# Hyperparameters
num_steps = 1000
betas = torch.linspace(0.0001, 0.02, steps=num_steps)
batch_size = 8
num_epochs = 5
device = check_cuda()

# DataLoader
dataset = TensorDataset(x_train)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Model, optimizer
model = UNet(input_channels=1, num_steps=num_steps).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

CUDA is not available. Running on CPU.


In [39]:
def tweedie_multi_sample(x_noisy, model, betas, num_steps, num_samples=10, device='cpu'):
    """
    Multi-sample Tweedie's formula for posterior mean estimation in diffusion models.

    Args:
        x_noisy (torch.Tensor): Noisy observation at step t, shape [batch, channels, H, W].
        model (nn.Module): Trained noise prediction model.
        betas (torch.Tensor): Noise schedule, shape [num_steps].
        num_steps (int): Total number of diffusion steps.
        num_samples (int): Number of reverse samples to average.
        device (torch.device): Device for computation.

    Returns:
        torch.Tensor: Estimated posterior mean (denoised sample).
    """
    batch_size = x_noisy.shape[0]
    t = torch.full((batch_size,), num_steps - 1, dtype=torch.long, device=device)
    beta_t = betas[t].view(-1, 1, 1, 1).to(device)

    # Collect multiple samples
    samples = []
    for _ in range(num_samples):
        predicted_noise = model(x_noisy, t)
        x0_hat = (x_noisy - torch.sqrt(beta_t) * predicted_noise) / torch.sqrt(1 - beta_t)
        samples.append(x0_hat.detach().cpu())
    samples = torch.stack(samples, dim=0)  # [num_samples, batch, channels, H, W]

    # Average over samples
    tweedie_estimate = samples.mean(dim=0)  # [batch, channels, H, W]
    return tweedie_estimate

In [40]:
# Assume model is trained, and you have a noisy sample x_noisy

# Example: get a batch of noisy data from your test set
x_noisy = torch.randn(4, 1, 72, 144).to(device)  # shape: [batch, channels, H, W]

# Use Tweedie's formula to estimate the denoised sample
denoised_sample = tweedie_multi_sample(
    x_noisy, model, betas, num_steps, num_samples=10, device=device
)

print("Denoised sample shape:", denoised_sample.shape)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x4 and 1000x256)

In [37]:
train_diffusion_model(model, dataloader, betas, num_steps, optimizer, device)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x8 and 1000x256)

In [ ]:
# # Training loop
# model.train()
# for epoch in range(num_epochs):
#     for batch in dataloader:
#         x = batch[0].to(device).to(torch.float32)  # Move data to the appropriate device
#         t = torch.randint(0, num_steps, (x.size(0),), device=device)  # Random timestep for each sample
#         beta_t = betas[t].view(-1, 1, 1, 1).to(device).to(torch.float32)
#         noise = torch.randn_like(x).to(torch.float32)  # Generate Gaussian noise
#         x_noisy = torch.sqrt(1 - beta_t) * x + torch.sqrt(beta_t) * noise  # Apply forward diffusion
#         print(x_noisy)
#         predicted_noise = model(x_noisy, t)
#         loss = diffusion_loss(predicted_noise, noise)
#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
#     print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

tensor([[[[-1.8708, -1.6447, -1.7812,  ..., -5.4288, -5.6211, -5.3776],
          [-1.8543, -1.7774, -1.6927,  ..., -5.5404, -5.5131, -5.3728],
          [-1.2279, -1.1562, -1.3371,  ..., -4.3402, -4.1519, -4.2678],
          ...,
          [ 4.6943,  4.4835,  4.5293,  ..., 10.7954, 10.9806, 10.8756],
          [ 5.8323,  5.9258,  5.7413,  ..., 10.4368, 10.2884, 10.3364],
          [ 6.9061,  6.4724,  7.0236,  ..., 10.2614, 10.1485, 10.1347]],

         [[-2.5867, -2.5144, -2.5345,  ..., -3.0834, -2.9455, -2.8716],
          [-1.1747, -1.5903, -1.4554,  ..., -2.8642, -2.8021, -2.7959],
          [-2.8140, -2.9474, -3.0106,  ..., -3.1654, -3.2310, -3.1910],
          ...,
          [ 4.7450,  4.9336,  4.7713,  ...,  7.0604,  7.0762,  7.2180],
          [ 4.5111,  4.2562,  4.4922,  ...,  7.7744,  7.7700,  7.9230],
          [ 6.0158,  6.2089,  6.0743,  ...,  7.4371,  7.4168,  7.4998]],

         [[-1.5646, -1.4032, -1.6056,  ..., -4.4930, -4.4257, -4.4232],
          [-2.2240, -2.0143, -

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x8 and 1000x256)